# Global CO2 Emissions & Climate Trends

**Data Source:** [Our World in Data](https://github.com/owid/co2-data) - Open-access CO2 and greenhouse gas emissions dataset

**Objective:** Analyze global emissions trends, identify top emitters, examine per-capita disparities, and track progress toward climate goals.

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (14,6), 'axes.titlesize': 14, 'axes.titleweight': 'bold'})

In [ ]:
df = pd.read_csv('../data/owid_co2_data.csv')
df['year'] = df['year'].astype(int)

# Convert numeric cols
num_cols = ['population','gdp','co2','co2_per_capita','co2_per_gdp','coal_co2','oil_co2','gas_co2',
            'cement_co2','co2_growth_prct','share_global_co2','cumulative_co2','energy_per_capita',
            'methane','nitrous_oxide','total_ghg']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Separate countries from aggregates
aggregates = ['World','Asia','Europe','North America','South America','Africa','Oceania']
world = df[df['country'] == 'World'].copy()
countries = df[~df['country'].isin(aggregates) & df['iso_code'].notna()].copy()

print(f'Total rows: {len(df):,}')
print(f'Countries: {countries["country"].nunique()}')
print(f'Year range: {df["year"].min()}-{df["year"].max()}')
df.head()

## 2. Global Emissions Trend

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,6))

# Total CO2 over time
axes[0].plot(world['year'], world['co2'], color='#C44E52', lw=2.5)
axes[0].fill_between(world['year'], world['co2'], alpha=0.2, color='#C44E52')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('CO2 Emissions (Mt)')
axes[0].set_title('Global CO2 Emissions Over Time')

# Year-over-year growth
axes[1].bar(world['year'], world['co2_growth_prct'], 
            color=['#55A868' if x < 0 else '#C44E52' for x in world['co2_growth_prct'].fillna(0)])
axes[1].axhline(y=0, color='black', lw=0.5)
axes[1].set_xlabel('Year'); axes[1].set_ylabel('YoY Growth (%)')
axes[1].set_title('Annual CO2 Growth Rate')

plt.tight_layout()
plt.savefig('../outputs/global_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Top Emitting Countries

In [ ]:
latest_year = countries['year'].max()
latest = countries[countries['year'] == latest_year].copy()
top15 = latest.nlargest(15, 'co2')[['country','co2','co2_per_capita','share_global_co2','population']]

fig, axes = plt.subplots(1, 2, figsize=(16,6))

# Total emissions
top15_sorted = top15.sort_values('co2', ascending=True)
axes[0].barh(top15_sorted['country'], top15_sorted['co2'], color='#C44E52', alpha=0.8)
axes[0].set_xlabel('CO2 Emissions (Mt)'); axes[0].set_title(f'Top 15 CO2 Emitters ({latest_year})')

# Per capita
top15_pc = top15.sort_values('co2_per_capita', ascending=True)
axes[1].barh(top15_pc['country'], top15_pc['co2_per_capita'], color='#DD8452', alpha=0.8)
axes[1].set_xlabel('CO2 per Capita (t)'); axes[1].set_title(f'Per Capita Emissions - Top 15 Emitters ({latest_year})')

plt.tight_layout()
plt.savefig('../outputs/top_emitters.png', dpi=150, bbox_inches='tight')
plt.show()
print(top15[['country','co2','co2_per_capita','share_global_co2']].to_string(index=False))

## 4. Emissions by Fuel Source

In [ ]:
fuels = world[['year','coal_co2','oil_co2','gas_co2','cement_co2']].dropna()

fig, ax = plt.subplots(figsize=(14,6))
ax.stackplot(fuels['year'], fuels['coal_co2'], fuels['oil_co2'], fuels['gas_co2'], fuels['cement_co2'],
             labels=['Coal','Oil','Gas','Cement'], alpha=0.8,
             colors=['#2C2C2C','#C44E52','#4C72B0','#DD8452'])
ax.set_xlabel('Year'); ax.set_ylabel('CO2 (Mt)'); ax.set_title('Global Emissions by Fuel Source')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/fuel_sources.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Per Capita Disparities

In [ ]:
# Compare top emitters per capita with population size
latest_valid = latest.dropna(subset=['co2_per_capita','population'])
latest_valid = latest_valid[latest_valid['population'] > 5e6]  # countries with >5M people

fig, ax = plt.subplots(figsize=(12,7))
scatter = ax.scatter(latest_valid['gdp']/latest_valid['population'], latest_valid['co2_per_capita'],
                     s=latest_valid['population']/2e6, alpha=0.5, edgecolors='black', lw=0.5)

for _, r in latest_valid.nlargest(10, 'co2').iterrows():
    if pd.notna(r['gdp']):
        ax.annotate(r['country'], (r['gdp']/r['population'], r['co2_per_capita']),
                    fontsize=8, textcoords='offset points', xytext=(5,5))

ax.set_xlabel('GDP per Capita ($)'); ax.set_ylabel('CO2 per Capita (tonnes)')
ax.set_title('CO2 per Capita vs GDP per Capita (size = population)')
plt.tight_layout()
plt.savefig('../outputs/percapita_vs_gdp.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Regional Emissions Trend

In [ ]:
regions = df[df['country'].isin(['Asia','Europe','North America','South America','Africa','Oceania'])]
region_trend = regions.pivot_table(index='year', columns='country', values='co2')

fig, ax = plt.subplots(figsize=(14,6))
for col in region_trend.columns:
    ax.plot(region_trend.index, region_trend[col], lw=2, label=col)
ax.set_xlabel('Year'); ax.set_ylabel('CO2 (Mt)'); ax.set_title('CO2 Emissions by Region')
ax.legend(); plt.tight_layout()
plt.savefig('../outputs/regional_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Cumulative Emissions (Historical Responsibility)

In [ ]:
cum = latest.nlargest(10, 'cumulative_co2')[['country','cumulative_co2']].dropna()
cum['cum_Gt'] = cum['cumulative_co2'] / 1000  # Mt to Gt

fig, ax = plt.subplots(figsize=(10,6))
cum_sorted = cum.sort_values('cum_Gt', ascending=True)
ax.barh(cum_sorted['country'], cum_sorted['cum_Gt'], color='#8172B2', alpha=0.8)
ax.set_xlabel('Cumulative CO2 (Gt)'); ax.set_title('Historical Cumulative CO2 Emissions (Top 10)')
for i, (_, r) in enumerate(cum_sorted.iterrows()):
    ax.text(r['cum_Gt']+0.5, i, f'{r["cum_Gt"]:,.0f} Gt', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/cumulative_emissions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Findings & Insights

1. **Global emissions continue rising** despite climate pledges, though growth rate has slowed
2. **Massive per-capita gap** between developed and developing nations - top emitters per capita are often small, wealthy nations
3. **Coal remains the #1 source** but gas is growing fastest - the "bridge fuel" narrative is playing out
4. **Asia now dominates** total emissions, but North America and Europe hold the largest cumulative (historical) responsibility
5. **GDP-emissions link persists** but some wealthy nations are decoupling (falling per-capita emissions while GDP grows)
6. **COVID-19 dip visible** in 2020 data, but emissions rebounded sharply in 2021-2022

### Policy Implications
- Per-capita metrics tell a very different story than totals - both matter for equitable climate policy
- Cumulative emissions data supports the "common but differentiated responsibilities" framework
- The gap between pledges and actual emissions trends suggests current policies are insufficient

---
*Data: Our World in Data (OWID) - CC BY license*